In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import xesmf as xe

In [ ]:
ds_ec = xr.open_dataset("EC-Earth3.grid.nc")
ds_era5 = xr.open_dataset(sorted((Path.home() / "ml-ds_data").glob("*.nc"))[0])
ds_carra2 = xr.open_dataset(sorted((Path.home() / "CARRA").glob("*.nc"))[0])

In [ ]:
coarse_lon = ds_ec["lon"].values
# Shift longitudes to the range [-180, 180] - for era5
coarse_lon = ((coarse_lon + 180) % 360) - 180
coarse_lon = np.sort(coarse_lon)
coarse_lat = ds_ec["lat"].values

In [ ]:
# Interpolate ERA5 -> EC-Earth3 grid (coarse)
era5_t2m_coarse = ds_era5.t2m.interp(
    latitude=coarse_lat,
    longitude=coarse_lon,
    method="linear",
)
# Remove areas with NaNs from era5_coarse
era5_t2m_coarse = era5_t2m_coarse.dropna(dim='latitude', how='all').dropna(dim='longitude', how='all')

In [ ]:
# Regrid ERA5 coarse t2m -> CARRA curvilinear grid
def _pick_coord(ds, candidates):
    for name in candidates:
        if name in ds.coords:
            return ds.coords[name]
        if name in ds:
            return ds[name]
    raise KeyError(f"None of {candidates} found in dataset")

carra_lon = _pick_coord(ds_carra2, ["longitude", "lon"])
carra_lat = _pick_coord(ds_carra2, ["latitude", "lat"])

# xESMF expects source grid names lon/lat
src = era5_t2m_coarse.rename({"longitude": "lon", "latitude": "lat"})

# Use lazy dask chunks to avoid allocating the full regridded time stack in memory
if "valid_time" in src.dims:
    src = src.chunk({"valid_time": 1})

# Curvilinear target grid can be provided as 2D lon/lat arrays
grid_out = xr.Dataset({"lon": carra_lon, "lat": carra_lat})

regridder = xe.Regridder(src, grid_out, method="bilinear", periodic=False, reuse_weights=False)
era5_t2m_on_carra = regridder(src, keep_attrs=True, output_chunks={"valid_time": 1})

era5_t2m_on_carra